In [1]:
# 01. THƯ VIỆN VÀ CẤU HÌNH
from pathlib import Path
from itertools import combinations
import hashlib
import json
import platform
import warnings

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import spearmanr
from sklearn.neighbors import LocalOutlierFactor

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
CANDIDATES = [
    ROOT / '03_model_ready_development.csv',
    ROOT / 'outputs/03_feature_engineering/03_model_ready_development.csv',
    Path.cwd() / '03_model_ready_development.csv',
    Path.cwd() / 'outputs/03_feature_engineering/03_model_ready_development.csv',
]
INPUT_PATH = next((p for p in CANDIDATES if p.is_file()), CANDIDATES[0])
PREVIOUS_CANDIDATES = [
    ROOT / '05_val_select_scores.csv',
    ROOT / 'outputs/05_model_training_A/05_val_select_scores.csv',
    Path.cwd() / '05_val_select_scores.csv',
    Path.cwd() / 'outputs/05_model_training_A/05_val_select_scores.csv',
]
PREVIOUS_PATH = next((p for p in PREVIOUS_CANDIDATES if p.is_file()), None)
OUTPUT_DIR = ROOT / 'outputs/06_3_lof_validation_A'
FEATURES = ['z_gap', 'z_intraday', 'z_range', 'z_volume']
EXPECTED_SHA256 = '0f035c54869b7389548ce6ce0b574335ed149e8378b09351eaa8245b840f472a'
K_OPTIONS = [10, 20, 35, 50, 100]
# Mỗi k thử trên Euclid. Manhattan chỉ đối chiếu tại k=20.
CONFIGS = [(k, 2) for k in K_OPTIONS] + [(20, 1)]
BASE_K, BASE_P = 20, 2
TOP_RATES = [0.01, 0.02, 0.05]


def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def n_top(n_rows, rate):
    return int(np.ceil(n_rows * rate))


def top_ids(scores, n):
    return set(np.argsort(-np.asarray(scores), kind='stable')[:n].tolist())


def rank_diagnostics(a, b, rates=TOP_RATES):
    a, b = np.asarray(a), np.asarray(b)
    result = {'spearman': float(spearmanr(a, b).statistic)}
    for rate in rates:
        n = n_top(len(a), rate)
        a_top, b_top = top_ids(a, n), top_ids(b, n)
        overlap = len(a_top & b_top)
        result[f'top{int(rate * 100)}_overlap_count'] = overlap
        result[f'top{int(rate * 100)}_overlap_pct'] = float(100.0 * overlap / n)
    return result


def fit_lof(k, p, X_train, X_val):
    assert 1 < k < len(X_train)
    model = LocalOutlierFactor(
        n_neighbors=k, metric='minkowski', p=p,
        novelty=True, contamination='auto', n_jobs=1,
    )
    model.fit(X_train)
    # score_samples chỉ chấm các phiên mới trong Validation-select.
    scores = -model.score_samples(X_val)
    if not np.isfinite(scores).all():
        raise AssertionError(f'Điểm LOF không hữu hạn: k={k}, p={p}')
    return model, np.asarray(scores, dtype=float)

In [2]:
# 02. ĐỌC TRAIN VÀ VAL-SELECT
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy Bước 03: {INPUT_PATH.resolve()}')
input_sha = sha256_file(INPUT_PATH)
if input_sha != EXPECTED_SHA256:
    raise ValueError('Dữ liệu khác phiên bản Bước 05/06.1. Kiểm tra file trước khi so sánh. '
                     f'SHA256 hiện tại={input_sha}')

source = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
required = ['ticker', 'date', 'split', 'val_role', *FEATURES]
assert set(required).issubset(source.columns), 'Thiếu cột Bước 03'
assert source[required].notna().all().all(), 'Có giá trị thiếu'
assert not source.duplicated(['ticker', 'date']).any(), 'Trùng ticker/date'
assert set(source['split']) == {'train', 'val'}, 'File chứa nhãn Test hoặc thiếu Train/Val'
assert set(source.loc[source['split'].eq('val'), 'val_role']) == {'select', 'calibrate'}
# Chỉ sử dụng Train và Val-select; không đưa Calibrate vào phân tích mô hình.
allowed = source['split'].eq('train') | (
    source['split'].eq('val') & source['val_role'].eq('select')
)
dev = source.loc[allowed, required].copy()
del source
for c in FEATURES:
    dev[c] = pd.to_numeric(dev[c], errors='raise')
assert np.isfinite(dev[FEATURES].to_numpy(float)).all(), 'Đặc trưng không hữu hạn'
dev['date'] = pd.to_datetime(dev['date'], format='%Y-%m-%d')
train = dev.loc[dev['split'].eq('train')].sort_values(['date', 'ticker']).reset_index(drop=True)
val = dev.loc[dev['split'].eq('val')].sort_values(['date', 'ticker']).reset_index(drop=True)
assert len(train) == 10387 and len(val) == 1430, f'Số dòng khác dự kiến: {len(train)}, {len(val)}'
assert train['date'].max() < val['date'].min(), 'Trùng thời gian Train/Val'
assert set(train['ticker']) == set(val['ticker']), 'Danh sách mã khác nhau'
X_train = train[FEATURES].to_numpy(dtype=np.float32)
X_val = val[FEATURES].to_numpy(dtype=np.float32)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Đã đọc:', len(train), 'Train;', len(val), 'Val-select. Không dùng Calibrate/Test.')

Đã đọc: 10387 Train; 1430 Val-select. Không dùng Calibrate/Test.


In [3]:
# 03. BASELINE VÀ ĐỐI CHIẾU
baseline_model, baseline = fit_lof(BASE_K, BASE_P, X_train, X_val)
checks = {'previous_scores_found': bool(PREVIOUS_PATH is not None),
          'source_sha256_correct': True,
          'n_train': int(len(train)), 'n_val_select': int(len(val))}
if PREVIOUS_PATH is not None:
    prev = pd.read_csv(PREVIOUS_PATH, encoding='utf-8-sig')
    assert {'ticker','date','score_lof'}.issubset(prev.columns)
    assert not prev.duplicated(['ticker','date']).any()
    cmp = val[['ticker','date']].copy()
    cmp['date'] = cmp['date'].dt.strftime('%Y-%m-%d')
    cmp['score_new'] = baseline
    merged = cmp.merge(prev[['ticker','date','score_lof']], on=['ticker','date'],
                       how='inner', validate='one_to_one')
    assert len(merged) == len(val), 'Điểm Bước 05 không đủ phiên hoặc khác thứ tự'
    diff = np.abs(merged['score_new'].to_numpy() - merged['score_lof'].to_numpy())
    checks['max_abs_difference_from_step05'] = float(diff.max())
    checks['match_step05_within_1e-5'] = bool(np.allclose(diff, 0, atol=1e-5, rtol=0))
    if not checks['match_step05_within_1e-5']:
        warnings.warn('Điểm LOF khác Bước 05; chưa được dùng để so sánh. Kiểm tra môi trường và input.')
else:
    checks['max_abs_difference_from_step05'] = None
    checks['match_step05_within_1e-5'] = None
print('Kiểm tra tái lập Bước 05:', checks)

baseline_table = val[['ticker','date',*FEATURES]].copy()
baseline_table['date'] = baseline_table['date'].dt.strftime('%Y-%m-%d')
baseline_table['score_lof'] = baseline
baseline_table['rank_lof'] = pd.Series(baseline).rank(ascending=False, method='first').astype(int)
baseline_table.to_csv(OUTPUT_DIR/'06_3_lof_baseline_val_scores.csv', index=False, encoding='utf-8-sig')

Kiểm tra tái lập Bước 05: {'previous_scores_found': True, 'source_sha256_correct': True, 'n_train': 10387, 'n_val_select': 1430, 'max_abs_difference_from_step05': 5.931091306976555e-08, 'match_step05_within_1e-5': True}


In [4]:
# 04. THỬ K VÀ P
models = {(BASE_K, BASE_P): baseline_model}
scores = {(BASE_K, BASE_P): baseline}
comparison_rows = []
for k, p in CONFIGS:
    if (k,p) not in scores:
        model, s = fit_lof(k, p, X_train, X_val)
        models[(k,p)], scores[(k,p)] = model, s
    s = scores[(k,p)]
    m = rank_diagnostics(baseline, s)
    comparison_rows.append({
        'n_neighbors':k, 'minkowski_p':p,
        'distance':'euclidean' if p==2 else 'manhattan',
        'score_median':float(np.median(s)),
        'score_p99':float(np.quantile(s,.99)),
        'score_max':float(np.max(s)),
        **m,
    })
config_df = pd.DataFrame(comparison_rows).sort_values(['minkowski_p','n_neighbors'], ascending=[False,True])
config_df.to_csv(OUTPUT_DIR/'06_3_lof_config_comparison.csv', index=False, encoding='utf-8-sig')

pair_rows = []
for (ka,pa),(kb,pb) in combinations(scores,2):
    pair_rows.append({
        'config_a':f'k{ka}_p{pa}', 'config_b':f'k{kb}_p{pb}',
        **rank_diagnostics(scores[(ka,pa)],scores[(kb,pb)])
    })
pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(OUTPUT_DIR/'06_3_lof_pairwise_config.csv', index=False, encoding='utf-8-sig')
print('\nKết quả thay k / metric so với baseline (không phải accuracy):')
print(config_df[['n_neighbors','distance','spearman','top1_overlap_pct','top5_overlap_pct']]
      .round(3).to_string(index=False))


Kết quả thay k / metric so với baseline (không phải accuracy):
 n_neighbors  distance  spearman  top1_overlap_pct  top5_overlap_pct
          10 euclidean     0.845            80.000            86.111
          20 euclidean     1.000           100.000           100.000
          35 euclidean     0.932            53.333            86.111
          50 euclidean     0.891            40.000            83.333
         100 euclidean     0.820            40.000            80.556
          20 manhattan     0.956            73.333            93.056


In [5]:
# 05. PHÂN TÍCH TOP-k
val_base = baseline_table.copy()
val_base['month'] = val_base['date'].str[:7]
by_date, by_month, by_ticker, top_blocks = [], [], [], []
for rate in TOP_RATES:
    n = n_top(len(val), rate)
    ids = np.argsort(-baseline, kind='stable')[:n]
    top = val_base.iloc[ids].copy()
    top.insert(2,'top_rate',rate)
    top_blocks.append(top)
    for date,count in top.groupby('date').size().items():
        by_date.append({'top_rate':rate,'date':date,'n_tickers':int(count)})
    val_month = val_base.groupby('month').size()
    top_month = top.groupby('month').size()
    for month,number in val_month.items():
        selected = int(top_month.get(month,0))
        by_month.append({'top_rate':rate,'month':month,
                         'n_val_rows':int(number),'n_top_rows':selected,'n_top_total':int(n),
                         'val_month_share_pct':float(100*number/len(val)),
                         'top_month_share_pct':float(100*selected/n),
                         'month_within_top_pct':float(100*selected/number)})
    val_tickers = val_base.groupby('ticker').size()
    top_tickers = top.groupby('ticker').size()
    for ticker,number in val_tickers.items():
        selected = int(top_tickers.get(ticker,0))
        by_ticker.append({'top_rate':rate,'ticker':ticker,
                          'n_val_rows':int(number),'n_top_rows':selected,
                          'top_share_within_ticker_pct':float(100*selected/number)})

date_df = pd.DataFrame(by_date).sort_values(['top_rate','n_tickers','date'], ascending=[True,False,True])
month_df = pd.DataFrame(by_month)
ticker_df = pd.DataFrame(by_ticker)
top_df = pd.concat(top_blocks,ignore_index=True)
for fname,table in [
    ('06_3_lof_concentration_by_date.csv',date_df),
    ('06_3_lof_concentration_by_month.csv',month_df),
    ('06_3_lof_concentration_by_ticker.csv',ticker_df),
    ('06_3_lof_top_scored_sessions.csv',top_df),
]:
    table.to_csv(OUTPUT_DIR/fname,index=False,encoding='utf-8-sig')
print('\nNhững ngày có nhiều mã thuộc LOF Top 5%:')
print(date_df[date_df['top_rate'].eq(.05)].head(8).to_string(index=False))


Những ngày có nhiều mã thuộc LOF Top 5%:
 top_rate       date  n_tickers
     0.05 2025-04-10         12
     0.05 2025-04-03          9
     0.05 2025-04-04          9
     0.05 2025-04-09          9
     0.05 2025-04-08          7
     0.05 2025-04-11          5
     0.05 2025-04-22          4
     0.05 2025-07-29          2


In [6]:
# 06. CHẨN ĐOÁN GAP VÀ HÀNG XÓM
assoc_rows = []
for feature in FEATURES:
    assoc_rows.append({'feature': feature,
                       'spearman_abs_feature_vs_lof':float(spearmanr(np.abs(val[feature]), baseline).statistic)})
assoc_df = pd.DataFrame(assoc_rows)
assoc_df.to_csv(OUTPUT_DIR/'06_3_lof_abs_feature_score_associations.csv', index=False, encoding='utf-8-sig')

gap_extreme = val_base.loc[val_base['z_gap'].abs() > 30].copy()
for rate in [0.01,0.05]:
    gap_extreme[f'in_top_{int(rate*100)}pct'] = gap_extreme['rank_lof'] <= n_top(len(val),rate)
gap_extreme.to_csv(OUTPUT_DIR/'06_3_lof_gap_extreme_val_review.csv', index=False, encoding='utf-8-sig')

neighbor_dist, neighbor_ids = baseline_model.kneighbors(X_val, n_neighbors=BASE_K)
train_tickers = train['ticker'].to_numpy()
val_tickers = val['ticker'].to_numpy()
same_ticker_pct = 100 * (train_tickers[neighbor_ids] == val_tickers[:,None]).mean(axis=1)
neighbor_df = pd.DataFrame({'ticker':val_tickers,'date':val_base['date'],
                            'same_ticker_neighbors_pct':same_ticker_pct,
                            'nearest_distance':neighbor_dist[:,0],
                            'kth_neighbor_distance':neighbor_dist[:,-1]})
neighbor_df.to_csv(OUTPUT_DIR/'06_3_lof_neighbor_composition_by_session.csv', index=False, encoding='utf-8-sig')
neighbor_ticker_df = neighbor_df.groupby('ticker',as_index=False).agg(
    n_rows=('date','size'),
    avg_same_ticker_pct=('same_ticker_neighbors_pct','mean'),
    median_kth_neighbor_distance=('kth_neighbor_distance','median')
)
neighbor_ticker_df.to_csv(OUTPUT_DIR/'06_3_lof_neighbor_composition_by_ticker.csv', index=False, encoding='utf-8-sig')

In [7]:
# 07. LƯU VÀ TẢI LẠI BASELINE
# Cùng dữ liệu/cấu hình, LOF không ngẫu nhiên.
re_model, re_scores = fit_lof(BASE_K, BASE_P, X_train, X_val)
checks['repeat_fit_max_abs_diff'] = float(np.max(np.abs(re_scores-baseline)))
assert np.allclose(baseline,re_scores,atol=1e-10,rtol=0)

model_path = OUTPUT_DIR/'06_3_lof_baseline_k20_euclidean.joblib'
# Chỉ tạo file nếu chưa có: không âm thầm ghi đè bản cũ.
if not model_path.exists():
    joblib.dump(baseline_model,model_path)
    checks['model_file_created'] = True
else:
    checks['model_file_created'] = False
    print('GIỮ NGUYÊN mô hình đã có, không ghi đè:', model_path)
# Lưu ý: chỉ load file joblib mà bạn tin cậy. Đây là file do chính script tạo ra.
# Chỉ tải file từ thư mục kết quả do chính script tạo / bạn tin cậy.
# Kể cả khi đã có file từ lần trước, vẫn xác minh nó khớp mô hình hiện tại.
restored = joblib.load(model_path)
restored_scores = -restored.score_samples(X_val)
checks['reload_max_abs_diff'] = float(np.max(np.abs(restored_scores-baseline)))
if not np.allclose(restored_scores, baseline, atol=1e-10, rtol=0):
    raise ValueError('File mô hình có sẵn KHÔNG khớp baseline hiện tại; không được dùng file này cho Bước 07')

In [8]:
# 08. BIỂU ĐỒ VÀ BÁO CÁO
plt.figure(figsize=(10,4.8))
k_df = config_df.loc[config_df['minkowski_p'].eq(2)].sort_values('n_neighbors')
plt.plot(k_df['n_neighbors'],k_df['top1_overlap_pct'],marker='o',label='Top 1%')
plt.plot(k_df['n_neighbors'],k_df['top5_overlap_pct'],marker='o',label='Top 5%')
plt.xlabel('Số hàng xóm k (Euclid)')
plt.ylabel('Trùng nhóm điểm cao so với k=20 (%)')
plt.title('LOF A — Độ nhạy thứ hạng khi thay số hàng xóm')
plt.ylim(0,105)
plt.legend()
plt.grid(alpha=.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'06_3_lof_k_sensitivity.png',dpi=160)
plt.close()

m5 = month_df.loc[month_df['top_rate'].eq(.05)].sort_values('month')
plt.figure(figsize=(9,4.8))
plt.bar(m5['month'],m5['n_top_rows'])
plt.xticks(rotation=40,ha='right')
plt.ylabel('Số phiên trong LOF Top 5%')
plt.title('LOF A — Phân bố nhóm điểm cao theo tháng (Val-select)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'06_3_lof_top5_by_month.png',dpi=160)
plt.close()

config = {
    'step':'06_3_lof_validation_A','role':'LOF là mô hình đối chiếu; IF là mô hình chính',
    'input_file': str(INPUT_PATH), 'input_sha256':input_sha,
    'train_n':int(len(train)),'val_select_n':int(len(val)),
    'features':FEATURES,'W':60,'scaler':'Step03 rolling median/MAD; no second scaler',
    'baseline':{'n_neighbors':20,'metric':'minkowski','p':2,'novelty':True,'contamination':'auto'},
    'configurations':[{'n_neighbors':k,'p':p} for k,p in CONFIGS],
    'top_rates':TOP_RATES,'test_and_calibrate_used':False,
    'technical_checks':checks,
    'versions':{'python':platform.python_version(),'pandas':pd.__version__,
                'numpy':np.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__},
    'limits':[
        'Không có nhãn bất thường thật: không suy ra Accuracy/Precision/Recall/F1.',
        'Top-k phục vụ phân tích, không phải ngưỡng cảnh báo đã hiệu chỉnh.',
        'LOF novelty=True: score_samples chỉ cho phiên mới, không so với negative_outlier_factor_ Train.',
        'So sánh thứ hạng và nhóm top-k; không so trực tiếp điểm với IF/AE.',
        'Thử số hàng xóm và khoảng cách không chứng minh cấu hình nào chính xác nhất.',
        'Chưa thử thay W, mức sàn hay biến đổi OHLCV (dành cho Bước 06.4).',
    ],
}
(OUTPUT_DIR/'06_3_lof_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')

# Markdown viết trực tiếp, không cần thư viện tabulate.
base = config_df[(config_df.n_neighbors==BASE_K)&(config_df.minkowski_p==BASE_P)].iloc[0]
alt = config_df[(config_df.n_neighbors==20)&(config_df.minkowski_p==1)].iloc[0]
month_apr = m5.loc[m5['month'].eq('2025-04')]
apr_count = int(month_apr.iloc[0]['n_top_rows']) if len(month_apr) else 0
apr_den = n_top(len(val),.05)

report = [
    '# Bước 06.3 — Kiểm chứng Local Outlier Factor (phương án A)',
    '',
    '**IF là mô hình chính. LOF là mô hình đối chiếu.** Không sử dụng Test, Val-calibrate hay nhãn proxy.',
    f'- Đầu vào: {len(train):,} Train; {len(val):,} Val-select; 4 đặc trưng chuẩn hóa W=60.',
    '- Baseline LOF: k=20, Euclid, `novelty=True`; điểm càng lớn càng khác thường.',
    f'- Tái lập Bước 05: {checks.get("match_step05_within_1e-5")} ; max diff: {checks.get("max_abs_difference_from_step05")}.',
    f'- Refit cùng cấu hình cho điểm trùng nhau: max diff = {checks["repeat_fit_max_abs_diff"]}.',
    '',
    '## So sánh cấu hình với baseline (chỉ độ nhất quán)',
    '',
    '| k | Khoảng cách | Spearman | Trùng Top 1% | Trùng Top 5% |',
    '|---:|---|---:|---:|---:|',
]
for _,r in config_df.iterrows():
    report.append(f'| {int(r.n_neighbors)} | {r.distance} | {r.spearman:.4f} | {r.top1_overlap_pct:.1f}% | {r.top5_overlap_pct:.1f}% |')
report += [
    '',
    '## Các dấu hiệu cần xem xét',
    f'- Trong Val-select: {len(gap_extreme)} phiên có |z_gap| > 30, trong đó {int(gap_extreme["in_top_5pct"].sum())} phiên thuộc LOF Top 5%.',
    f'- Top 5% có {apr_count}/{apr_den} phiên ở tháng 04/2025 (chưa có diễn giải nguyên nhân).',
    f'- Trung bình tỷ lệ hàng xóm cùng mã: {float(np.mean(same_ticker_pct)):.2f}% (LOF dùng một mô hình gộp 13 mã).',
    '- Các mốc Top 1/2/5% và |z_gap| > 30 chỉ phục vụ kiểm tra, KHÔNG phải ngưỡng bất thường chính thức.',
    '',
    '## Quyết định ở giai đoạn này',
    '- Giữ LOF k=20, Euclid làm baseline để so sánh, CHƯA chốt cấu hình cuối.',
    '- Không chọn k/metric chỉ dựa vào Spearman hoặc độ trùng Top-k.',
    '- Tiếp tục Bước 06.4 (thử thay đổi OHLCV hợp lệ, W, mức sàn) sau khi tổng hợp LOF với IF/AE.',
    '- Không huấn luyện lại bằng Val-calibrate hoặc Test; chưa tính ngưỡng cảnh báo.',
    '',
    '## Các cảnh báo phương pháp',
    '- Hàng xóm của LOF nằm trong Train của 13 mã gộp, không nhất thiết thuộc cùng mã.',
    '- Điểm LOF cho Val-select là novelty scoring; điểm trên Train nếu dùng negative_outlier_factor_ không so trực tiếp.',
    '- LOF nhạy với cách đo khoảng cách và cách chuẩn hóa; điểm lớn không tự chứng minh bất thường thật.',
]
(OUTPUT_DIR/'06_3_lof_report.md').write_text('\n'.join(report)+'\n',encoding='utf-8')
print('\n== HOÀN THÀNH BƯỚC 06.3 ==')
print('Tệp kết quả tại:',OUTPUT_DIR.resolve())
print('Top 5% tháng 04/2025:',apr_count,'/',apr_den)
print('Kiểm chứng lưu mô hình:',checks)


== HOÀN THÀNH BƯỚC 06.3 ==
Tệp kết quả tại: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\06_3_lof_validation_A
Top 5% tháng 04/2025: 58 / 72
Kiểm chứng lưu mô hình: {'previous_scores_found': True, 'source_sha256_correct': True, 'n_train': 10387, 'n_val_select': 1430, 'max_abs_difference_from_step05': 5.931091306976555e-08, 'match_step05_within_1e-5': True, 'repeat_fit_max_abs_diff': 0.0, 'model_file_created': True, 'reload_max_abs_diff': 0.0}
